In [ ]:
# import pandas as pd
# import numpy as np
# from sklearn.preprocessing import StandardScaler
# from sklearn.cluster import DBSCAN
# from scipy.spatial.distance import cdist
# from xgboost import XGBRegressor

# # ۱. خواندن فایل اصلی
# file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
# output_filename = r'outputs\G11\dsas_g11_bearings_vibration_temp_deviation_monitoring\deviation_monitoring\dsas_g11_bearings_vibration_temp_deviation_monitoring_output2.xlsx'
# try:
#     df_raw = pd.read_excel(file_path)
#     print("فایل اصلی با موفقیت خوانده شد.")
# except Exception as e:
#     print(f"خطا: {e}")
#     exit()

# # لیست فیچرها و تارگت‌ها
# all_features = [
#      'AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361', 
#     'AssetID_9368', 'AssetID_9369', 'AssetID_9370', 'AssetID_9357',
#     'AssetID_9343', 'AssetID_9344', 'AssetID_9408'
# ]
# target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']

# # ۲. حذف نویز و داده‌های پرت (DBSCAN)
# scaler = StandardScaler()
# scaled_data = scaler.fit_transform(df_raw[all_features])
# dbscan = DBSCAN(eps=0.5, min_samples=5)
# labels = dbscan.fit_predict(scaled_data)

# cluster_centers = {i: scaled_data[labels == i].mean(axis=0) for i in set(labels) if i != -1}
# def calc_dist(idx):
#     label = labels[idx]
#     point = scaled_data[idx].reshape(1, -1)
#     if label != -1:
#         return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
#     return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0

# df_raw['distance'] = [calc_dist(i) for i in range(len(df_raw))]
# df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
# df_cleaned['date'] = pd.to_datetime(df_cleaned['date'])
# df_cleaned = df_cleaned.sort_values(by='date')

# # ۳. آموزش مدل و پیش‌بینی
# split_idx = int(len(df_cleaned) * 0.80)
# train_df = df_cleaned.iloc[:split_idx].copy()
# test_df = df_cleaned.iloc[split_idx:].copy()

# for target in target_sensors:
#     X_train, y_train = train_df[[f for f in all_features if f != target]], train_df[target]
#     model = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
#     model.fit(X_train, y_train)
#     # ایجاد ستون پیش‌بینی با نام موقت برای Unpivot کردن راحت‌تر
#     test_df[f'PRED_{target}'] = model.predict(test_df[[f for f in all_features if f != target]])

# # ۴. تغییر ساختار داده (Unpivot/Melt) به فرمت درخواستی شما
# # مرحله اول: استخراج مقادیر واقعی (Actual)
# melted_actual = test_df.melt(id_vars=['date'], value_vars=target_sensors, 
#                              var_name='AssetID', value_name='actual')

# # مرحله دوم: استخراج مقادیر پیش‌بینی شده (Predicted)
# pred_cols = [f'PRED_{t}' for t in target_sensors]
# melted_pred = test_df.melt(id_vars=['date'], value_vars=pred_cols, 
#                            var_name='temp_target', value_name='predicted')


# # اصلاح نام AssetID در دیتافریم پیش‌بینی برای Merge کردن (حذف PRED_)
# melted_pred['AssetID'] = melted_pred['temp_target'].str.replace('PRED_', '')

# # مرحله سوم: ادغام دو دیتافریم بر اساس تاریخ و نام سنسور
# final_long_df = pd.merge(melted_actual, melted_pred[['date', 'AssetID', 'predicted']], 
#                          on=['date', 'AssetID'])

# # ۵. ذخیره خروجی نهایی
# final_long_df.to_excel(output_filename, index=False)

# print("فایل جدید با ساختار مورد نظر شما (date, AssetID, actual, predicted) ایجاد شد.")


فایل اصلی با موفقیت خوانده شد.
فایل جدید با ساختار مورد نظر شما (date, AssetID, actual, predicted) ایجاد شد.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import cdist
from xgboost import XGBRegressor
import os
import time
from datetime import datetime

# غیرفعال کردن هشدارهای غیرضروری
import warnings
warnings.filterwarnings('ignore')

def run_deviation_monitoring():
    """اجرای تحلیل پایش انحراف با XGBoost و ذخیره خروجی (فرمت Long)"""
    
    print("="*60)
    print(f"🔄 شروع تحلیل در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*60)
    
    # ۱. خواندن فایل اصلی
    file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
    output_filename = r'outputs\G11\dsas_g11_bearings_vibration_temp_deviation_monitoring\deviation_monitoring\dsas_g11_bearings_vibration_temp_deviation_monitoring_output2.xlsx'
    
    # ایجاد پوشه خروجی
    os.makedirs(os.path.dirname(output_filename), exist_ok=True)
    
    try:
        df_raw = pd.read_excel(file_path)
        print(f"✅ فایل اصلی با موفقیت خوانده شد. تعداد رکوردها: {len(df_raw):,}")
        print(f"📅 بازه زمانی: {df_raw['date'].min()} تا {df_raw['date'].max()}")
    except Exception as e:
        print(f"❌ خطا: {e}")
        return None

    # لیست فیچرها و تارگت‌ها
    all_features = [
        'AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361', 
        'AssetID_9368', 'AssetID_9369', 'AssetID_9370', 'AssetID_9357',
        'AssetID_9343', 'AssetID_9344', 'AssetID_9408'
    ]
    target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']

    # ۲. حذف نویز و داده‌های پرت (DBSCAN)
    print("🔄 مرحله 1: پیش‌پردازش و حذف داده‌های پرت...")
    
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_raw[all_features])
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    labels = dbscan.fit_predict(scaled_data)

    cluster_centers = {i: scaled_data[labels == i].mean(axis=0) for i in set(labels) if i != -1}
    
    def calc_dist(idx):
        label = labels[idx]
        point = scaled_data[idx].reshape(1, -1)
        if label != -1:
            return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
        return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0

    before_count = len(df_raw)
    df_raw['distance'] = [calc_dist(i) for i in range(len(df_raw))]
    df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
    after_count = len(df_cleaned)
    
    print(f"   حذف {before_count - after_count:,} ردیف به عنوان داده‌های پرت")
    
    df_cleaned['date'] = pd.to_datetime(df_cleaned['date'])
    df_cleaned = df_cleaned.sort_values(by='date')

    # ۳. آموزش مدل و پیش‌بینی
    print("🔄 مرحله 2: یادگیری و پیش‌بینی برای تمام سنسورها...")
    
    split_idx = int(len(df_cleaned) * 0.80)
    train_df = df_cleaned.iloc[:split_idx].copy()
    test_df = df_cleaned.iloc[split_idx:].copy()
    
    print(f"   داده‌های آموزش: {len(train_df):,} رکورد")
    print(f"   داده‌های تست: {len(test_df):,} رکورد")

    for target in target_sensors:
        features = [f for f in all_features if f != target]
        X_train, y_train = train_df[features], train_df[target]
        
        model = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, verbosity=0)
        model.fit(X_train, y_train)
        
        # ایجاد ستون پیش‌بینی با نام موقت برای Unpivot کردن راحت‌تر
        test_df[f'PRED_{target}'] = model.predict(test_df[features])
        
        # محاسبه RMSE
        mse = np.mean((test_df[target] - test_df[f'PRED_{target}']) ** 2)
        rmse = np.sqrt(mse)
        print(f"   ✅ {target}: RMSE = {rmse:.6f}")

    # ۴. تغییر ساختار داده (Unpivot/Melt) به فرمت درخواستی شما
    print("🔄 مرحله 3: تغییر ساختار داده به فرمت Long...")
    
    # مرحله اول: استخراج مقادیر واقعی (Actual)
    melted_actual = test_df.melt(id_vars=['date'], value_vars=target_sensors, 
                                 var_name='AssetID', value_name='actual')

    # مرحله دوم: استخراج مقادیر پیش‌بینی شده (Predicted)
    pred_cols = [f'PRED_{t}' for t in target_sensors]
    melted_pred = test_df.melt(id_vars=['date'], value_vars=pred_cols, 
                               var_name='temp_target', value_name='predicted')

    # اصلاح نام AssetID در دیتافریم پیش‌بینی برای Merge کردن (حذف PRED_)
    melted_pred['AssetID'] = melted_pred['temp_target'].str.replace('PRED_', '')

    # مرحله سوم: ادغام دو دیتافریم بر اساس تاریخ و نام سنسور
    final_long_df = pd.merge(melted_actual, melted_pred[['date', 'AssetID', 'predicted']], 
                             on=['date', 'AssetID'])
    
    # اضافه کردن ستون خطا (Error)
    final_long_df['error'] = final_long_df['actual'] - final_long_df['predicted']
    final_long_df['abs_error'] = np.abs(final_long_df['error'])
    
    print(f"   ✅ تعداد رکوردهای نهایی: {len(final_long_df):,}")

    # ۵. ذخیره خروجی نهایی
    print("💾 مرحله 4: ذخیره خروجی...")
    
    try:
        final_long_df.to_excel(output_filename, index=False)
        print(f"✅ فایل جدید با ساختار مورد نظر ایجاد شد: {output_filename}")
        print(f"📊 تعداد رکوردها: {len(final_long_df):,}")
        print(f"📋 ستون‌ها: date, AssetID, actual, predicted, error, abs_error")
        
        # نمایش آمار خطاها
        print("\n📊 آمار خطاها:")
        print(f"   میانگین خطا: {final_long_df['error'].mean():.6f}")
        print(f"   میانگین خطای مطلق: {final_long_df['abs_error'].mean():.6f}")
        print(f"   حداکثر خطای مطلق: {final_long_df['abs_error'].max():.6f}")
        
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل: {e}")
        return None
    
    print("="*60)
    print(f"✅ تحلیل در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} کامل شد")
    print("="*60)
    
    return final_long_df

def run_scheduler():
    """
    بررسی مداوم برای اجرا در زمان‌های مشخص (هر روز)
    """
    print("="*60)
    print("🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - پایش انحراف بیرینگ با XGBoost (فرمت Long)")
    print("="*60)
    print("⏰ زمان‌های اجرا (هر روز):")
    print("   - ساعت 10:00")
    print("   - ساعت 10:05")
    print("   - ساعت 10:10")
    print("="*60)
    print("💡 برای توقف برنامه، Ctrl+C را بزنید")
    print("="*60)
    
    last_run_time = None  # فقط برای جلوگیری از اجرای مجدد در یک زمان
    
    while True:
        try:
            now = datetime.now()
            current_time = now.strftime("%H:%M")
            
            # بررسی زمان‌های مشخص
            if current_time in ["22:02", "22:03", "22:04"]:
                # فقط چک می‌کنیم که در همین زمان دوبار اجرا نشود
                if last_run_time != current_time:
                    print("\n" + "="*60)
                    print(f"⏰ زمان اجرا فرا رسید: {now.strftime('%Y-%m-%d %H:%M:%S')}")
                    print("="*60)
                    
                    # اجرای تابع اصلی
                    result = run_deviation_monitoring()
                    
                    if result is not None:
                        print("\n" + "="*60)
                        print("✅ اجرای زمان‌بندی شده با موفقیت کامل شد!")
                        print("="*60)
                    else:
                        print("\n" + "="*60)
                        print("❌ اجرای زمان‌بندی شده با شکست مواجه شد!")
                        print("="*60)
                    
                    # ثبت زمان اجرا
                    last_run_time = current_time
                    
                    # 10 ثانیه صبر کن تا از اجرای مجدد در همان دقیقه جلوگیری شود
                    time.sleep(10)
            
            # هر 10 ثانیه یکبار بررسی کن
            time.sleep(10)
            
        except KeyboardInterrupt:
            print("\n" + "="*60)
            print("⏹️ برنامه با دستور کاربر متوقف شد")
            print(f"⏹️ زمان توقف: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print("="*60)
            break
            
        except Exception as e:
            print(f"❌ خطا در حلقه اصلی: {e}")
            print("🔄 ادامه اجرا...")
            time.sleep(60)

# اجرای اصلی
if __name__ == "__main__":
    try:
        print("="*60)
        print("🚀 شروع برنامه پایش انحراف بیرینگ با XGBoost (فرمت Long)")
        print("="*60)
        
        # شروع زمان‌بندی
        run_scheduler()
        
    except Exception as e:
        print(f"❌ خطای غیرمنتظره: {e}")
        input("برای خروج Enter بزنید...")

🚀 شروع برنامه پایش انحراف بیرینگ با XGBoost (فرمت Long)
🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - پایش انحراف بیرینگ با XGBoost (فرمت Long)
⏰ زمان‌های اجرا (هر روز):
   - ساعت 10:00
   - ساعت 10:05
   - ساعت 10:10
💡 برای توقف برنامه، Ctrl+C را بزنید

⏰ زمان اجرا فرا رسید: 2026-07-01 22:02:34
🔄 شروع تحلیل در 2026-07-01 22:02:34
✅ فایل اصلی با موفقیت خوانده شد. تعداد رکوردها: 11,752
📅 بازه زمانی: 2021-03-16 05:33:48 تا 2026-05-04 05:16:35
🔄 مرحله 1: پیش‌پردازش و حذف داده‌های پرت...
   حذف 1,175 ردیف به عنوان داده‌های پرت
🔄 مرحله 2: یادگیری و پیش‌بینی برای تمام سنسورها...
   داده‌های آموزش: 8,461 رکورد
   داده‌های تست: 2,116 رکورد
   ✅ AssetID_9358: RMSE = 9.424736
   ✅ AssetID_9359: RMSE = 20.025093
   ✅ AssetID_9360: RMSE = 4.691303
   ✅ AssetID_9361: RMSE = 11.051948
🔄 مرحله 3: تغییر ساختار داده به فرمت Long...
   ✅ تعداد رکوردهای نهایی: 8,488
💾 مرحله 4: ذخیره خروجی...
✅ فایل جدید با ساختار مورد نظر ایجاد شد: outputs\G11\dsas_g11_bearings_vibration_temp_deviation_monitoring\deviation